# Food Delivery Time Prediction (improved version)
Steps: load -> inspect -> clean -> feature engineering -> EDA -> skewness check -> linear regression -> assumption checks -> conclusion.

## 1. Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer, TransformedTargetRegressor
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score

## 2. Data importing and first look

In [ ]:
# Keep the CSV in the same folder as this notebook (no hardcoded C:\\ path)
df = pd.read_csv("deliverytime.csv")

print("Rows, columns:", df.shape)
df.info()

In [ ]:
df.head()

In [ ]:
# describe() shows min / max / mean: the quickest way to spot impossible values
df.describe()

## 3. Data cleaning
Missing values and duplicates are not enough. We also check that the values are **possible**
(rating between 1 and 5, coordinates inside India, adult delivery person).

In [ ]:
print("Duplicate rows:", df.duplicated().sum())
print("Missing values:")
print(df.isna().sum())

In [ ]:
lat_cols = ["Restaurant_latitude", "Delivery_location_latitude"]
lon_cols = ["Restaurant_longitude", "Delivery_location_longitude"]

valid = (
    df[lat_cols].apply(lambda s: s.between(6, 38)).all(axis=1)      # India latitude range
    & df[lon_cols].apply(lambda s: s.between(68, 98)).all(axis=1)   # India longitude range
    & df["Delivery_person_Ratings"].between(1, 5)
    & (df["Delivery_person_Age"] >= 18)
)

print("Rows that look invalid:", (~valid).sum(), "of", len(df))
df = df[valid].copy()          # .copy() so we never edit a view by accident
print("Rows kept:", len(df))

## 4. Feature engineering
1. `city`: hidden inside `Delivery_person_ID` (e.g. `INDORES13DEL02` -> `INDO`).
2. `distance_km`: straight-line distance, calculated with the fast vectorised haversine formula.
3. Drop `ID` and `Delivery_person_ID`: unique-ish text IDs make the model memorise instead of learn.

In [ ]:
def haversine(lat1, lon1, lat2, lon2):
    # distance in km between two points on Earth; works on whole columns at once
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    a = (np.sin((lat2 - lat1) / 2) ** 2
         + np.cos(lat1) * np.cos(lat2) * np.sin((lon2 - lon1) / 2) ** 2)
    return 6371 * 2 * np.arcsin(np.sqrt(a))

df["distance_km"] = haversine(
    df["Restaurant_latitude"], df["Restaurant_longitude"],
    df["Delivery_location_latitude"], df["Delivery_location_longitude"],
)

df["city"] = df["Delivery_person_ID"].str.extract(r"^([A-Z]+?)RES")
print("Rows where city could not be extracted:", df["city"].isna().sum())
print(df["city"].value_counts())

df = df.drop(columns=["ID", "Delivery_person_ID"])
df["distance_km"].describe()

## 5. Exploratory data analysis

In [ ]:
target = "Time_taken(min)"

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.histplot(df[target], kde=True, ax=axes[0])
axes[0].set_title("Distribution of delivery time")
sns.boxplot(x=df[target], ax=axes[1])
axes[1].set_title("Delivery time boxplot")
plt.tight_layout()
plt.show()

In [ ]:
# Distance vs time: a scatter of a sample (NOT groupby on a continuous column)
sample = df.sample(min(5000, len(df)), random_state=0)
sns.scatterplot(data=sample, x="distance_km", y=target, alpha=0.3)
plt.title("Distance vs delivery time")
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
for ax, col in zip(axes, ["Type_of_vehicle", "Type_of_order", "city"]):
    df.groupby(col)[target].mean().sort_values().plot.bar(ax=ax)
    ax.set_title(f"Mean delivery time by {col}")
    ax.set_ylabel("minutes")
plt.tight_layout()
plt.show()

In [ ]:
num = ["Delivery_person_Age", "Delivery_person_Ratings", "distance_km"]

sns.heatmap(df[num + [target]].corr(), annot=True, cmap="coolwarm", fmt=".2f")
plt.title("Correlation (only meaningful numeric columns)")
plt.show()

## 6. Skewness check (look first, transform only if needed)
Rule of thumb: skew between -0.5 and +0.5 is fine. Linear regression assumes the **residuals** are
roughly normal, not that every column is. So we do **not** transform by default.

In [ ]:
print(df[num + [target]].skew())

## 7. Linear regression

In [ ]:
cat = ["Type_of_order", "Type_of_vehicle", "city"]

X = df[num + cat]
y = df[target]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=0
)

def make_preprocessor():
    return ColumnTransformer([
        ("num", StandardScaler(), num),
        ("cat", OneHotEncoder(handle_unknown="ignore"), cat),
    ])

pipe = Pipeline([
    ("prep", make_preprocessor()),
    ("lm", LinearRegression()),
])
pipe.fit(X_train, y_train)

In [ ]:
def rmse(y_true, y_pred):
    return np.sqrt(mean_squared_error(y_true, y_pred))

pred_train = pipe.predict(X_train)
pred_test = pipe.predict(X_test)

print(f"Train R2 : {r2_score(y_train, pred_train):.3f}")
print(f"Test  R2 : {r2_score(y_test, pred_test):.3f}")
print(f"Train RMSE (minutes): {rmse(y_train, pred_train):.2f}")
print(f"Test  RMSE (minutes): {rmse(y_test, pred_test):.2f}")

# Baseline: always predict the training average
baseline = np.full(len(y_test), y_train.mean())
print(f"Baseline RMSE (always guess the mean): {rmse(y_test, baseline):.2f}")

## 8. Check the assumptions on the final model
- Residuals vs predicted: should look like a shapeless cloud around 0.
- Q-Q plot / histogram: residuals should roughly follow a straight line / bell shape.

In [ ]:
resid = y_test - pred_test

fig, axes = plt.subplots(1, 3, figsize=(16, 4))
axes[0].scatter(pred_test, resid, alpha=0.2)
axes[0].axhline(0, color="red")
axes[0].set_xlabel("Predicted")
axes[0].set_ylabel("Residual")
axes[0].set_title("Residuals vs predicted")

stats.probplot(resid, plot=axes[1])
axes[1].set_title("Q-Q plot of residuals")

sns.histplot(resid, kde=True, ax=axes[2])
axes[2].set_title("Residual distribution")
plt.tight_layout()
plt.show()

## 9. What did the model learn? (coefficients)

In [ ]:
names = pipe.named_steps["prep"].get_feature_names_out()
coefs = pd.Series(pipe.named_steps["lm"].coef_, index=names).sort_values()
print(coefs)

## 10. Optional experiment: log-transform the target
`TransformedTargetRegressor` trains on log(time) but automatically converts predictions back to minutes,
so RMSE stays in minutes and can be compared fairly with the model above.

In [ ]:
pipe_log = Pipeline([
    ("prep", make_preprocessor()),
    ("lm", TransformedTargetRegressor(
        regressor=LinearRegression(), func=np.log1p, inverse_func=np.expm1)),
])
pipe_log.fit(X_train, y_train)
pred_log = pipe_log.predict(X_test)

print(f"Plain model     -> RMSE {rmse(y_test, pred_test):.2f} min, R2 {r2_score(y_test, pred_test):.3f}")
print(f"Log-target model -> RMSE {rmse(y_test, pred_log):.2f} min, R2 {r2_score(y_test, pred_log):.3f}")

## 11. Conclusion (fill this in with YOUR numbers)
- What I cleaned and why:
- Features I created:
- Test R2 and RMSE (minutes) compared with the baseline:
- Did the residual plots show problems?
- What I would try next (e.g. more features, other models):